# FIAP • ENGENHARIA DA COMPUTAÇÃO (Turma 1ECR • 2022)
## Disciplina: ARQUITETURA E ORGANIZAÇÃO DE COMPUTADORES
### Organização de Processadores e Arquitetura do Neander (Checkpoint 4)

> **Descrição do Projeto e Objetivos:**  
> Ciclo de Busca e Execução, Registradores PC, AC, RI, REM, RDM e Flags NZ.

---



# Checkpoint 4: Emulação do Ciclo de Instrução da CPU Neander
Simulação em software da arquitetura interna e registradores do processador.


In [ ]:
class NeanderCPU:
    def __init__(self):
        self.mem = [0] * 256
        self.pc = 0
        self.ac = 0
        self.flag_n = 0
        self.flag_z = 1
        self.halted = False
        
    def update_flags(self):
        self.flag_z = 1 if self.ac == 0 else 0
        self.flag_n = 1 if self.ac >= 128 else 0
        
    def step(self):
        if self.halted: return
        opcode = self.mem[self.pc]
        self.pc += 1
        if opcode == 0x20: # LDA
            addr = self.mem[self.pc]
            self.pc += 1
            self.ac = self.mem[addr]
            self.update_flags()
        elif opcode == 0x30: # ADD
            addr = self.mem[self.pc]
            self.pc += 1
            self.ac = (self.ac + self.mem[addr]) & 0xFF
            self.update_flags()
        elif opcode == 0x10: # STA
            addr = self.mem[self.pc]
            self.pc += 1
            self.mem[addr] = self.ac
        elif opcode == 0xF0: # HLT
            self.halted = True

# Carregando o programa na memoria
cpu = NeanderCPU()
cpu.mem[0x00] = 0x20; cpu.mem[0x01] = 0x10 # LDA 0x10
cpu.mem[0x02] = 0x30; cpu.mem[0x03] = 0x11 # ADD 0x11
cpu.mem[0x04] = 0x10; cpu.mem[0x05] = 0x12 # STA 0x12
cpu.mem[0x06] = 0xF0                       # HLT
cpu.mem[0x10] = 35
cpu.mem[0x11] = 48

while not cpu.halted:
    cpu.step()

print(f'Execução finalizada com sucesso!')
print(f'Conteúdo de Mem[0x12]: {cpu.mem[0x12]} (Esperado: 35 + 48 = 83)')
print(f'Acumulador AC: {cpu.ac}, Flags: N={cpu.flag_n}, Z={cpu.flag_z}')



In [ ]:
import matplotlib.pyplot as plt

ciclos = ['Início', 'LDA 0x10', 'ADD 0x11', 'STA 0x12', 'HLT']
ac_vals = [0, 35, 83, 83, 83]
pc_vals = [0, 2, 4, 6, 7]

fig, ax1 = plt.subplots(figsize=(8, 4))
color = '#1f77b4'
ax1.set_xlabel('Ciclo de Instrução da CPU', fontweight='bold')
ax1.set_ylabel('Valor do Acumulador AC', color=color, fontweight='bold')
ax1.step(ciclos, ac_vals, color=color, lw=2.5, where='mid', label='Acumulador (AC)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5, 100)

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('Contador de Programa (PC)', color=color, fontweight='bold')
ax2.plot(ciclos, pc_vals, color=color, marker='o', lw=2, linestyle='--', label='PC')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evolução Temporal dos Registradores PC e AC na CPU Neander', fontweight='bold')
ax1.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

